In [1]:
import cv2
import mediapipe as mp
import pyautogui
import numpy as np
import time
import math

cap = cv2.VideoCapture(0)
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

screen_width, screen_height = pyautogui.size()
frame_r = 150 

cv2.namedWindow("Virtual Mouse", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Virtual Mouse", 320, 240) 

plocX, plocY = 0, 0
clocX, clocY = 0, 0
smoothening = 5

while True:
    success, img = cap.read()
    if not success: break

    img = cv2.flip(img, 1)
    h, w, c = img.shape
    
    cv2.rectangle(img, (frame_r, frame_r), (w - frame_r, h - frame_r), (255, 0, 255), 2)

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    if results.multi_hand_landmarks:
        for hand_lms in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(img, hand_lms, mp_hands.HAND_CONNECTIONS)

            index_finger = hand_lms.landmark[8]  
            middle_finger = hand_lms.landmark[12] 
            thumb_finger = hand_lms.landmark[4]  

            ix, iy = int(index_finger.x * w), int(index_finger.y * h)
            mx, my = int(middle_finger.x * w), int(middle_finger.y * h)
            tx, ty = int(thumb_finger.x * w), int(thumb_finger.y * h)
            
            mouse_x = np.interp(ix, [frame_r, w - frame_r], [0, screen_width])
            mouse_y = np.interp(iy, [frame_r, h - frame_r], [0, screen_height])
            
            clocX = plocX + (mouse_x - plocX) / smoothening
            clocY = plocY + (mouse_y - plocY) / smoothening

            pyautogui.moveTo(clocX, clocY, _pause=False)

            plocX, plocY = clocX, clocY

            dist_single = math.hypot(ix - tx, iy - ty) 
            dist_double = math.hypot(mx - tx, my - ty) 

            if dist_single < 30:
                pyautogui.click()
                cv2.putText(img, "SINGLE CLICK!", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                time.sleep(0.3) 

            elif dist_double < 30:
                pyautogui.doubleClick()
                cv2.putText(img, "DOUBLE CLICK!", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                time.sleep(0.3) 

    cv2.imshow("Virtual Mouse", img)
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

C:\Users\User\.conda\envs\mp310\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
